# Tuning of the sampling parameters for LLMs (repetition penalty, temp, top_p, sys prompts, etc)

Il y a la description de chaque parametre dans la def de la fonction stv


In [73]:
from typing import List
import mlx_lm
from datetime import datetime

def load_and_chat_with_mlx(
    dir: str = "./data/models/33", # Directory path where the MLX model and tokenizer are stored. This is used to load the pre-trained model for generating text.
    temp: float = 0.1, # Sets the likeliness to choose less probable tokens. Lower values make the output more deterministic, while higher values increase randomness.
    top_p: float = 0.3, # Implements nucleus sampling, where the model considers only the smallest set of tokens whose cumulative probability exceeds top_p. This helps in generating more coherent and contextually relevant text.
    max_tokens: int = 3200, # The maximum number of tokens to generate in the response. This limits the length of the output and influences how detailed the response can be.
    min_p: float = 0.0, # A threshold to filter out tokens with very low probabilities, ensuring that only tokens with a probability above min_p are considered during sampling.
    top_k: int = 64, # Limits the sampling pool to the top_k most probable tokens, which can help in reducing randomness and focusing on more likely continuations.
    prompt_list: List = None, # A list of dictionaries representing the chat history, including system instructions and user messages. This provides context for the model to generate relevant responses.
    repetition_penalty: float = 1.3, # A penalty applied to tokens that have already been generated, discouraging the model from repeating the same phrases and promoting more diverse outputs.
    repetition_context_size: int = 60, # The size of the context window used to track token repetitions for applying the repetition penalty. A larger context size allows the model to consider a broader history when avoiding repetitions.
    min_new_tokens: int = 5, # The minimum number of new tokens that must be generated before the model is allowed to produce an end-of-sequence (EOS) token. This ensures that the response has a certain length before it can conclude.
    patience: int = 5, # The number of consecutive identical tokens that can be generated before the EOS restriction is lifted. This helps to prevent the model from getting stuck in loops of repetition
):

    print(f"Loading MLX model and tokenizer from {dir}...")
    start = datetime.now()
    mlx_model, mlx_tok = mlx_lm.load(dir)
    print(f"Model and tokenizer loaded in {datetime.now() - start}")

    # get EOS ids
    eos_ids = list(mlx_tok.eos_token_ids)
    print(f"eos_ids from tokenizer: {eos_ids}")

    # Sampler
    verbose = True
    sampler = mlx_lm.sample_utils.make_sampler(
        temp,
        top_p,
        min_p=min_p,
        top_k=top_k,
        xtc_special_tokens=mlx_tok.encode("\n") + eos_ids
    )

    # Tokenize prompt
    print("Tokenizing prompt")
    start = datetime.now()
    prompt_tok = mlx_tok.apply_chat_template(
        prompt_list,
        tokenize=False,
        add_generation_prompt=True
    )
    print(f"Prompt is:\n\n{prompt_tok}")
    print(f"Prompt tokenized in {datetime.now() - start} seconds")
    prompt_tokens = mlx_tok.apply_chat_template(
        prompt_list, add_generation_prompt=True
    )
    
    prompt_len = len(prompt_tokens)
    print(f"Computed prompt length: {prompt_len}")

    # --- robust min-new-tokens processor ---
    def min_new_tokens_processor(min_new_tokens: int = 5, prompt_len: int = prompt_len, eos_ids=None, patience: int = 5):
        """
        Forbid EOS until at least `min_new_tokens` have been generated AFTER the prompt.
        If the model is stuck repeating the same token `patience` times, allow EOS to break out.
        """
        if eos_ids is None:
            local_eos_ids = []
        else:
            local_eos_ids = eos_ids

        def processor(tokens, logits):
            # tokens is an mx.array of the full sequence (prompt + generated)
            try:
                tokens_len = int(tokens.size)  # preferred for mx.array
            except Exception:
                try:
                    tokens_len = len(tokens)
                except Exception:
                    print(f"Failed to compute tokens length, got type {type(tokens)}")
                    tokens_len = 0

            generated = tokens_len - prompt_len
            # forbid EOS until we reach the minimum generated tokens
            if generated < min_new_tokens:
                for eid in local_eos_ids:
                    if eid is None:
                        continue
                    # safety: ensure index in vocab range
                    if 0 <= eid < logits.shape[-1]:
                        logits[:, eid] = -1e9 # What does this mean ? Why -1e9 ?

            # simple stuck-detection: if last `patience` tokens are identical, allow EOS to escape
            # (prevents infinite loops where the model just repeats one token)
            if generated >= 1 and generated >= patience:
                try:
                    # tokens.tolist() -> list of ints
                    last = tokens.tolist()[-patience:]
                    if len(last) == patience and all(x == last[0] for x in last):
                        # do nothing -> EOS allowed (we do not re-apply -1e9)
                        pass
                except Exception:
                    # if tolist() fails for some reason, ignore
                    pass

            return logits

        return processor

    # --- Build logits_processors (keep repetition_penalty) and append our min_new_tokens processor ---
    logits_processors = mlx_lm.sample_utils.make_logits_processors(
        repetition_penalty=repetition_penalty,
        repetition_context_size=repetition_context_size,
    )

    logits_processors.append(min_new_tokens_processor(min_new_tokens=min_new_tokens, prompt_len=prompt_len, eos_ids=eos_ids, patience=patience))


    # Generate response
    print(f"Generating response from MLX model for question: {prompt_list[-1]['content']}")
    start = datetime.now()
    response = mlx_lm.generate(
        mlx_model,
        mlx_tok,
        prompt_tokens,
        max_tokens=max_tokens,
        verbose=verbose,
        sampler=sampler,
        logits_processors=logits_processors,
        prompt_cache=None
    )
    del mlx_model, mlx_tok  # free memory
    print(f"Response generated in {datetime.now() - start} seconds")


In [82]:
system_prompt = f"""You are a concise and helpful assistant; answer directly in the user's tone without repeating context or mentioning instructions."""
prompt_list = [
    {"role": "user", "content": system_prompt + "\n\n" + input("Enter the prompt") },
]

temp = 0.2
top_p = 0.5
max_tokens = 1200
min_p = 0.0
top_k = 64
repetition_penalty = 1.3
repetition_context_size = 60
min_new_tokens = 5
patience = 5

print("1B")
print("="*40)
load_and_chat_with_mlx(
    dir="./data/models/q4_manuel",
    temp=temp,
    top_p=top_p,
    max_tokens=max_tokens,
    min_p=min_p,
    top_k=top_k,
    prompt_list=prompt_list,
    repetition_penalty=repetition_penalty,
    repetition_context_size=repetition_context_size,
    min_new_tokens=min_new_tokens,
    patience=patience
)
print("="*40)
print("2B")
print("="*40)
load_and_chat_with_mlx(
    dir="./data/models/34",
    temp=temp,
    top_p=top_p,
    max_tokens=max_tokens,
    min_p=min_p,
    top_k=top_k,
    prompt_list=prompt_list,
    repetition_penalty=repetition_penalty,
    repetition_context_size=repetition_context_size,
    min_new_tokens=min_new_tokens,
    patience=patience
)
print("="*40)
print("4B")
print("="*40)
load_and_chat_with_mlx(
    dir="./data/models/35",
    temp=temp,
    top_p=top_p,
    max_tokens=max_tokens,
    min_p=min_p,
    top_k=top_k,
    prompt_list=prompt_list,
    repetition_penalty=repetition_penalty,
    repetition_context_size=repetition_context_size,
    min_new_tokens=min_new_tokens,
    patience=patience
)
print("="*40)
print("7B")
print("="*40)
load_and_chat_with_mlx(
    dir="./data/models/36",
    temp=temp,
    top_p=top_p,
    max_tokens=max_tokens,
    min_p=min_p,
    top_k=top_k,
    prompt_list=prompt_list,
    repetition_penalty=repetition_penalty,
    repetition_context_size=repetition_context_size,
    min_new_tokens=min_new_tokens,
    patience=patience
)
print("="*40)

1B
Loading MLX model and tokenizer from ./data/models/q4_manuel...
Model and tokenizer loaded in 0:00:02.829815
eos_ids from tokenizer: [1, 106]
Tokenizing prompt
Prompt is:

<bos><start_of_turn>user
You are a concise and helpful assistant; answer directly in the user's tone without repeating context or mentioning instructions.

Hey how are you?<end_of_turn>
<start_of_turn>model

Prompt tokenized in 0:00:00.013088 seconds
Computed prompt length: 38
Generating response from MLX model for question: You are a concise and helpful assistant; answer directly in the user's tone without repeating context or mentioning instructions.

Hey how are you?
Good! I’m doing well, thanks for asking. How about you? 😊 

Let me know if there's anything specific you need help with today.
👍
Do you want to tell me something or would you like a task?
Prompt: 38 tokens, 139.092 tokens-per-sec
Generation: 52 tokens, 103.247 tokens-per-sec
Peak memory: 8.199 GB
Response generated in 0:00:00.826929 seconds
2B
Load